
# PS2: Mini-batch SGD and CNN

Advanced Learning 2026/7.



---


Imports:

In [1]:
import numpy as np # You are allowed to use  only numpy.
import time, sys, cProfile
import matplotlib.pyplot as plt

In [4]:
from srcPS2 import *

Note that now all of the network's objects from PS1 are loaded into the workspace.

---

### 0. Loading data

As in PS1, you will evaluate your home-made network on the `mnist` dataset.   
The MNIST dataset is a large dataset of handwritten digits that is commonly used for training various image and vision models.  



In [5]:
from keras.datasets import mnist
from keras.utils import to_categorical
# load MNIST from server
# Using a standard library (keras.datasets) to load the mnist data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


---

## PS2 Part 1: Mini-batch SGD



**Welcome back**.   






Mini-batch Stochastic Gradient Descent (Mini-batch SGD) is an optimization algorithm used to train machine learning models, particularly neural networks. It strikes a balance between two other methods:

    Batch Gradient Descent – Uses the entire dataset to compute gradients and update the model parameters.
        ✅ Accurate gradients
        ❌ Slow and memory-intensive for large datasets

    Stochastic Gradient Descent (SGD) – Uses a single data point to compute gradients for each update.
        ✅ Fast and efficient
        ❌ Noisy updates, which can lead to instability

  
**Mini-batch SGD**.

Mini-batch SGD computes gradients and updates the model parameters using small, randomly selected batches of data (typically 32, 64, or 128 samples).

    ✅ Faster than batch gradient descent
    ✅ Less noisy than pure SGD
    ✅ Efficient use of memory
    ✅ Enables vectorization for faster computation on GPUs

**How it Works**.

    Shuffle the dataset to avoid bias.

    Divide the data into small batches (e.g., 64 samples per batch).

    For each epoch, iterate through the mini-batches:
        Perform forward pass to calculate predictions.
        Compute the loss for the mini-batch.
        Perform backpropagation to compute gradients.
        Update model parameters using the gradients.

    Repeat until the model converges.

**Algorithm**

For each mini-batch $B={x_1,x_2,...,x_m}B={x_1​,x_2​,...,x_m​}$ of size $m$, update the weights $w$ as:    

$$
w=w−η⋅\frac{1}{m}∑_{i=1}^m ∇L(w,x_i)
$$

Where:   
*  $η$ = learning rate.
* $∇L(w,x_i)$ = gradient of the loss for sample $x_i$.   

Advantages

    Efficient and scalable – Works well with large datasets.
    Smooth convergence – Less noisy than SGD but faster than full batch training.
    Parallelization – Batches can be processed in parallel on GPUs.

Disadvantages

    Choosing batch size can affect performance. Small batches lead to noise, large batches consume more memory.
    Convergence may be slower than batch gradient descent for some problems.


### Mini-batch Questions.
**P1Q1**:  

Please answer the following questions providing statistical justifications and mathematical formulas when needed.

P1Q1a: Explain how the mini-batch size impacts convergence, model performance, and computational efficiency.



**Answer (P1Q1a).**

Let the training loss be $L(w)=\frac1N\sum_{i=1}^N \ell_i(w)$. A mini-batch $B$ of size $m$ gives the estimate
$\hat g_B=\frac1m\sum_{i\in B}\nabla \ell_i(w)$, which is **unbiased**, $\mathbb E[\hat g_B]=\nabla L(w)$, with covariance
$\mathrm{Cov}(\hat g_B)\approx \frac{1}{m}\Sigma$ (for $m\ll N$; exactly $\frac{N-m}{N-1}\frac{\Sigma}{m}$ when sampling without replacement),
where $\Sigma$ is the covariance of the per-sample gradients. So the gradient noise (its standard deviation) shrinks like $1/\sqrt m$: doubling the batch only reduces the noise by a factor $\sqrt2$.

* **Convergence.** Per *update*, larger batches give a more accurate direction, so a larger learning rate can be used and fewer updates are needed. Per *epoch* (one pass over the data), a batch of size $m$ gives only $N/m$ updates, and since the noise reduction is only $1/\sqrt m$ while the cost per update grows linearly in $m$, the gain in progress per sample seen diminishes beyond a "critical batch size" (the point where $\frac{1}{m}\mathrm{tr}\Sigma$ becomes small compared with $\|\nabla L\|^2$). Very small batches ($m=1$, the `fit` method of PS1) make many noisy updates; with a fixed step size the iterates keep fluctuating in a region whose size is proportional to $\eta\,\mathrm{tr}\Sigma/m$, so convergence to a precise minimum requires a decaying learning rate (see P1Q1b). A common heuristic is the *linear scaling rule*: when the batch size is multiplied by $k$, multiply $\eta$ by $k$ (with warm-up), which keeps $\eta/m$ - the effective "temperature" of SGD - constant.
* **Model performance (generalisation).** The gradient noise is not only a nuisance: it acts as an implicit regulariser. SGD with small batches tends to end up in *flat* minima, which generalise better, while very large batches tend to converge to *sharp* minima and show a "generalisation gap" (Keskar et al., 2017) unless the learning rate, warm-up and number of updates are re-tuned. Too small batches, on the other hand, can give such noisy gradients that training becomes unstable or stalls at a higher loss (the noise term $\eta\sigma^2/m$ dominates).
* **Computational efficiency.** The work per update is $O(m\cdot\text{cost of one forward/backward})$, but on vectorised hardware (BLAS/GPUs) a batch of $m$ samples is one matrix-matrix product instead of $m$ matrix-vector products, so the wall-clock time per update grows sub-linearly with $m$ until the hardware is saturated. Memory grows linearly with $m$ (all activations must be stored for back-propagation). Hence typical choices of 32-256: large enough to use the hardware and to get a usable gradient, small enough to keep many updates per epoch, the regularising noise, and a modest memory footprint.

**P1Q1b:**   
Why does mini-batch SGD require a learning rate schedule, and what are the common scheduling strategies?



**Answer (P1Q1b).**

With a constant learning rate $\eta$, mini-batch SGD does **not** converge to a minimiser but to a *stationary distribution* around it. For a quadratic loss $L(w)=\frac12 h w^2$ with gradient noise of variance $\sigma^2/m$, the update $w_{t+1}=w_t-\eta(hw_t+\xi_t)$ has stationary variance
$$\mathrm{Var}(w_\infty)=\frac{\eta\,\sigma^2/m}{h(2-\eta h)}\;\approx\;\frac{\eta\sigma^2}{2hm},$$
i.e. the final loss floor is proportional to $\eta/m$. Only by sending $\eta_t\to0$ does the noise vanish and the iterates settle in the minimum. The classical Robbins-Monro conditions for almost-sure convergence of stochastic approximation are
$$\sum_t \eta_t=\infty\quad\text{and}\quad\sum_t\eta_t^2<\infty,$$
(e.g. $\eta_t\propto 1/t$): the first guarantees that the algorithm can travel arbitrarily far, the second that the accumulated noise is finite. In practice a schedule also lets us start with a *large* step (fast progress through the flat, far-from-optimum region, escaping poor basins) and finish with a *small* step (fine convergence, lower loss floor). Full-batch GD has no noise term and could in principle use a fixed step, which is why the need for a schedule is specific to (mini-batch) SGD.

Common strategies:
* **Step decay** - multiply $\eta$ by a factor (e.g. 0.5 or 0.1) every $k$ epochs: $\eta_t=\eta_0\,\gamma^{\lfloor t/k\rfloor}$. (This is what `step_decay` in `srcPS2.py` implements and what is used below.)
* **Exponential decay** $\eta_t=\eta_0 e^{-kt}$, and **inverse / polynomial decay** $\eta_t=\eta_0/(1+kt)$ or $\eta_0/\sqrt t$ (satisfies Robbins-Monro).
* **Cosine annealing** $\eta_t=\eta_{\min}+\frac12(\eta_0-\eta_{\min})(1+\cos(\pi t/T))$, often with **warm restarts** (SGDR) / cyclical learning rates.
* **Warm-up**: increase $\eta$ linearly from a small value during the first few hundred updates, which stabilises the start when the batch is large or the weights are random; frequently combined with one of the decays above (e.g. the "one-cycle" policy).
* **Adaptive / performance-based**: reduce $\eta$ when the validation loss plateaus (`ReduceLROnPlateau`), or use per-parameter adaptive step sizes (AdaGrad, RMSProp, Adam), which rescale the step by running estimates of the gradient's second moment - though even Adam is usually combined with a decay schedule.

P1Q1c: What role does batch normalization play in mini-batch SGD, and why is it essential for deep neural networks?



**Answer (P1Q1c).**

Batch normalisation (Ioffe & Szegedy, 2015) normalises every pre-activation feature **using the statistics of the current mini-batch**:
$$\mu_B=\frac1m\sum_{i\in B}x_i,\qquad \sigma_B^2=\frac1m\sum_{i\in B}(x_i-\mu_B)^2,\qquad
\hat x_i=\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}},\qquad y_i=\gamma\hat x_i+\beta ,$$
with learnable scale/shift $\gamma,\beta$ so that the layer does not lose expressiveness. At test time the mini-batch statistics are replaced by running (exponential moving) averages collected during training, so BN is intrinsically tied to mini-batch SGD: it needs a batch of reasonable size ($m\gtrsim 16$-32) for $\mu_B,\sigma_B^2$ to be good estimates of the population statistics (for very small batches one uses LayerNorm / GroupNorm instead).

Why it is essential for deep networks:
* **Keeps the input distribution of every layer standardised** ("reduces internal covariate shift"): without it, the early layers' updates change the distribution seen by the later layers, which must constantly re-adapt. With BN the scale of activations is fixed regardless of the weights, so the gradient w.r.t. the weights is invariant to the weight scale ($\partial y/\partial w$ shrinks when $w$ grows), which prevents exploding/vanishing activations in networks with many layers.
* **Smoother optimisation landscape**: Santurkar et al. (2018) showed BN mainly improves the Lipschitz-ness of the loss and of its gradients (lower $\beta$-smoothness), so larger learning rates can be used, convergence is faster, and the result is far less sensitive to the weight initialisation.
* **Mitigates saturation** of sigmoid/tanh units and dead ReLUs, since the pre-activations are kept around zero with unit variance.
* **Regularisation**: because each sample is normalised with statistics of a *random* batch, its output is slightly noisy; this acts like a small amount of noise injection / dropout and improves generalisation.
In short, BN makes training of very deep nets (ResNets, etc.) with plain mini-batch SGD stable and fast, something that was very difficult before it.

P1Q1d: How does mini-batch SGD differ in performance compared to full-batch gradient descent when dealing with non-convex optimization problems?



**Answer (P1Q1d).**

For a **convex** problem both methods converge to the unique global minimum, and full-batch GD does so with a cleaner, monotone decrease of the loss (linear rate $O(\rho^t)$ for strongly convex problems versus $O(1/t)$ for SGD with a decaying step). Neural-network losses, however, are **non-convex**: they have many local minima, plateaus and, above all, a huge number of saddle points (in high dimensions a random critical point is overwhelmingly likely to be a saddle, with some negative Hessian eigenvalues).

* **Full-batch GD** is deterministic: $w_{t+1}=w_t-\eta\nabla L(w_t)$. It follows the exact gradient to the *nearest* stationary point of the basin in which it started. Near a saddle the gradient becomes very small, so GD slows down dramatically and can stay there for a very long time (it escapes only along directions in which it is already slightly displaced). It cannot climb out of a poor local minimum at all. Each step costs a full pass over the $N$ samples, so for large $N$ the number of steps that can be afforded is small.
* **Mini-batch SGD** replaces $\nabla L$ by $\hat g_B=\nabla L+\xi$, $\mathrm{Cov}(\xi)\approx\Sigma/m$. The noise term is exactly what helps in the non-convex case: it perturbs the iterate off the unstable manifold of a saddle (so the negative-curvature direction grows exponentially and the saddle is left quickly), and it lets the iterate hop out of narrow, sharp minima whose width is smaller than the noise amplitude $\propto\sqrt{\eta/m}$, while it is trapped only by wide, flat minima. These flat minima are precisely the ones that generalise well (the loss does not change much under the perturbation train-set $\to$ test-set). SGD also makes $N/m$ updates per epoch instead of one, so in terms of wall-clock time it reaches a low loss far earlier.

Empirically, for deep networks SGD with a moderate batch typically reaches a *lower training loss in a given compute budget* and *better test accuracy* than full-batch GD; full-batch GD tends to converge to sharper minima with a larger generalisation gap (and recent work shows it needs explicit noise injection or very careful tuning to match SGD). The trade-off is that SGD's loss curve is noisy, it needs a learning-rate schedule to converge precisely, and its result depends on the random batch order. In the experiment below the "GD" baseline is actually the extreme opposite case - *pure* SGD with $m=1$ - so the comparison is between maximum noise/maximum number of updates and the averaged, vectorised mini-batch version.

P1Q1e: How does mini-batch SGD perform when training on imbalanced datasets, and what strategies can mitigate its challenges?



**Answer (P1Q1e).**

If class $k$ has prior probability $p_k$, the number of its samples in a random mini-batch of size $m$ is $\mathrm{Binomial}(m,p_k)$, with mean $mp_k$ and the probability that the batch contains **no** sample of the class is $(1-p_k)^m$. For a 1 % minority class and $m=64$ this is $0.99^{64}\approx0.53$: more than half of the updates contain no information about the minority class, and the ones that do are dominated by the $\approx63$ majority samples. Consequently:
* the mini-batch gradient is almost entirely the majority-class gradient, so the model drifts towards predicting the majority class (the loss minimiser for a batch without minority samples is "always majority"), i.e. high accuracy but very poor recall for the rare class;
* the minority part of the gradient has very high variance between batches ($\mathrm{Var}\propto p_k(1-p_k)/m$ relative to the mean $p_k$), making training noisy and unstable for exactly the class we care about;
* the overall loss and "accuracy" look fine during training, which hides the problem - the standard accuracy is not a meaningful metric here.

Mitigation strategies:
* **Re-sampling the batches**: *stratified / class-balanced sampling* (each batch contains a fixed proportion of every class), *over-sampling* the minority class (with or without augmentation, e.g. SMOTE) or *under-sampling* the majority. This directly fixes the composition of every mini-batch.
* **Re-weighting the loss**: class weights $w_k\propto1/p_k$ (or the "effective number of samples"), so that in expectation each class contributes equally to $\hat g_B$: $\hat g_B=\frac1m\sum_{i\in B}w_{y_i}\nabla\ell_i$. Focal loss down-weights easy (majority) samples.
* **Larger batches** (or gradient accumulation) reduce $(1-p_k)^m$ and the variance of the minority gradient.
* **Decision-level fixes**: adjust the decision threshold / prior correction after training, and evaluate with precision-recall, F1, balanced accuracy or AUC instead of plain accuracy.
* **Two-stage / decoupled training**: train the representation on the imbalanced data, then re-train only the classifier head with balanced sampling.

P1Q1f: Explain why larger mini-batch sizes are preferred during inference but not necessarily during training.


**Answer (P1Q1f).**

At **inference** there is no gradient and no parameter update: the output for a sample depends only on that sample (with batch-norm in evaluation mode using fixed running statistics), so the *result* is identical for any batch size. The batch size then only affects **throughput**: a forward pass over $m$ samples is a single matrix-matrix product that keeps the GPU/BLAS pipeline busy, with the fixed overheads (kernel launches, Python-level loop, data transfer) amortised over $m$ samples. The cost per sample therefore decreases with $m$ until the hardware is saturated, and the only limits are memory (activations of $m$ samples, but no stored intermediate values for backprop and no gradient buffers, so $\approx 1/3$ of the training memory per sample) and, in on-line systems, latency - a request must not wait for a batch to fill up.

During **training** the batch size is a *hyper-parameter of the optimisation algorithm*, not just a performance knob (see P1Q1a):
* the gradient noise $\propto\sigma^2/m$ is part of the algorithm; removing it with huge batches changes the dynamics (sharp minima, generalisation gap) and must be compensated by re-tuning $\eta$, warm-up and the number of steps;
* for a fixed budget of samples processed, a larger batch means fewer parameter updates ($N/m$ per epoch), and beyond the critical batch size extra samples per step give almost no additional progress per step - the computation is wasted;
* memory grows linearly with $m$ *and* every activation must be stored for the backward pass, together with the gradient and optimiser state (e.g. Adam moments), so the feasible training batch is far smaller than the feasible inference batch.
So large batches are a pure win for inference (more samples per second, same predictions) but a trade-off for training (speed vs. number of updates vs. generalisation).

### Applying mini-batch SGD

Preparing the data:

In [6]:
# training data : 60000 samples
# reshape and normalize input data
x_train = x_train.reshape(x_train.shape[0], 1, 28*28)
x_train = x_train.astype('float32')
x_train /= 255
# One-hot encoding of the output.
# Currently a number in range [0,9]; Change into a vector of size 10
# e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
# same for test data : 10000 samples
x_test = x_test.reshape(x_test.shape[0], 1, 28*28)
x_test = x_test.astype('float32')
x_test /= 255
y_test = to_categorical(y_test)

P1Q2a:  

Add a new function to the class `MyNetwork` named `fit_mini_batch`. This function should be similar to `fit`, just with a mini-batch SGD implementation. You can also make other code changes to the scripts in `srcPS2` if you think they are needed. In addition, add a suitable learning rate scheduler of your choice.


YOU SHOULD (a) add the missing script to the srcPS2 code, and (b) make a copy of the function  below for review:

In [7]:
### Copy of the code added to srcPS2.py (for review) ###

# Learning-rate schedulers for mini-batch SGD.
# Each returns a callable schedule(epoch) -> lr  (epoch is 0-based), which can be
# passed directly as the `learning_rate` argument of MyNetwork.fit_mini_batch.
def step_decay(lr0, drop=0.5, epochs_drop=5):
    """
    Step decay: the learning rate is multiplied by `drop` every `epochs_drop` epochs.
        lr(t) = lr0 * drop ** floor(t / epochs_drop)
    """
    return lambda epoch: lr0 * drop ** np.floor(epoch / epochs_drop)


def exponential_decay(lr0, k=0.1):
    """
    Exponential decay:  lr(t) = lr0 * exp(-k t)
    """
    return lambda epoch: lr0 * np.exp(-k * epoch)


def cosine_decay(lr0, total_epochs, lr_min=0.0):
    """
    Cosine annealing:  lr(t) = lr_min + 0.5 (lr0 - lr_min) (1 + cos(pi t / T))
    """
    return lambda epoch: lr_min + 0.5 * (lr0 - lr_min) * (1 + np.cos(np.pi * epoch / total_epochs))


class MyNetwork(MyNetwork):   # (in srcPS2.py this is simply a method of MyNetwork)

    def fit_mini_batch(self, x_train, y_train, batch_size, epochs, learning_rate,
                       shuffle=True, verbose=True):
        """
        Trains the network using mini-batch stochastic gradient descent.

        One parameter update is performed per mini-batch (instead of per sample
        as in `fit`).  The whole batch is pushed through the layers as a single
        matrix, so the forward/backward passes are vectorised over the batch.
        The provided `mse_grad` divides by `y_true.size`, so the gradient that
        reaches the layers is already the *mean* over the batch
        (w <- w - lr * 1/m * sum_i grad L(w, x_i)), exactly as in the
        mini-batch SGD formula.

        Parameters:
          x_train : np.array
              Training data, shape (N, 1, D) or (N, D).
          y_train : np.array
              Training labels, shape (N, 1, K), (N, K) or (N,).
          batch_size : int
              Size of each mini-batch.
          epochs : int
              Number of training epochs.
          learning_rate : float or callable
              Either a constant learning rate, or a learning-rate *scheduler*:
              a callable `lr = schedule(epoch)` (epoch is 0-based) - see
              `step_decay`, `exponential_decay`, `cosine_decay` below.
          shuffle : bool
              Re-shuffle the data at the start of every epoch (default True).
          verbose : bool
              Print the average training error (and lr) of every epoch.

        Returns:
          list of the average training error per epoch.
        """
        samples = len(x_train)
        # A (N,1,D) sample layout (as used by the per-sample `fit`) is squeezed
        # to (N,D) so that a batch is a plain 2-D matrix (B,D) for the affine layers.
        X = x_train.reshape(samples, -1) if x_train.ndim == 3 and x_train.shape[1] == 1 else x_train
        Y = y_train.reshape(samples, -1) if y_train.ndim >= 2 else y_train
        # Accept a constant learning rate or a schedule(epoch) callable.
        schedule = learning_rate if callable(learning_rate) else (lambda epoch: learning_rate)

        n_batches = int(np.ceil(samples / batch_size))
        if verbose:
            print("Training on {:,} samples, {} mini-batches of size {}:".format(
                samples, n_batches, batch_size))
        history = []
        for i in range(epochs):
            lr = schedule(i)
            idx_all = np.random.permutation(samples) if shuffle else np.arange(samples)
            err = 0
            for b in range(n_batches):
                idx = idx_all[b * batch_size:(b + 1) * batch_size]
                xb, yb = X[idx], Y[idx]
                # forward pass on the whole mini-batch
                output = xb
                for layer in self.layers:
                    output = layer.forward_propagation(output)
                # loss (for display only) - weighted by batch size for the epoch mean
                err += self.loss(yb, output) * len(idx)
                # backward pass: one parameter update per mini-batch
                grad = self.loss_grad(yb, output)
                for layer in reversed(self.layers):
                    grad = layer.backward_propagation(grad, lr)
            err /= samples
            history.append(err)
            if verbose:
                print("Training epoch %d/%d   lr=%.5f   error=%f" % (i + 1, epochs, lr, err))
        return history


    Test the two networks below to compare the differences between GD and mini-batch SGD:

In [8]:
#### GD implementaiton ####

# Network Architecture
netGD = MyNetwork()

netGD.add(Affine_Layer(28*28, 128))
netGD.add(ActivationLayer(tanh, tanh_grad))
netGD.add(Affine_Layer(128, 64))
netGD.add(ActivationLayer(tanh, tanh_grad))
netGD.add(Affine_Layer(64, 10))
netGD.add(ActivationLayer(tanh, tanh_grad))

netGD.use_loss(mse, mse_grad)

epoch_num = 10
lr = 0.05
t1 = time.time()
netGD.fit(x_train[:10000], y_train[:10000], epochs=epoch_num, learning_rate=lr)
print(f"Total process time: {round(time.time() - t1,3)}")

Training on 10,000 samples:
Training epoch 1/10   error=0.041548
Training epoch 2/10   error=0.023433
Training epoch 3/10   error=0.018082
Training epoch 4/10   error=0.015214
Training epoch 5/10   error=0.013316
Training epoch 6/10   error=0.011911
Training epoch 7/10   error=0.010798
Training epoch 8/10   error=0.009874
Training epoch 9/10   error=0.009080
Training epoch 10/10   error=0.008379
Total process time: 50.501


In [9]:
output_GD = netGD.predict(x_test ,y_test )


Test set: Avg. loss: 0.0113, Accuracy: 9419/10000 (94%)



In [10]:
#### mini-batch SGD implementaiton ####

# Network Architecture
netMiniGD = MyNetwork()

netMiniGD.add(Affine_Layer(28*28, 128))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))
netMiniGD.add(Affine_Layer(128, 64))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))
netMiniGD.add(Affine_Layer(64, 10))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))

netMiniGD.use_loss(mse, mse_grad)

epoch_num = 10
# Learning-rate scheduler: step decay, lr(t) = 2.0 * 0.5**floor(t/3)  ->  2.0, 2.0, 2.0, 1.0, 1.0, 1.0, 0.5, ...
# NOTE: mse_grad divides by y_true.size, so the batch gradient that reaches the layers is the *mean*
# over the 128 samples; one update per batch therefore needs a much larger step than the
# per-sample GD above (which does 128x more updates). 2.0 is the starting value that worked best
# among {0.5, 1, 2, 4} in a short sweep.
lr_sched = step_decay(lr0=2.0, drop=0.5, epochs_drop=3)
t2 = time.time()
netMiniGD.fit_mini_batch(x_train[:10000], y_train[:10000], batch_size=128, epochs=epoch_num, learning_rate=lr_sched)
print(f"Total process time: {round(time.time() - t2,3)}")

Training on 10,000 samples, 79 mini-batches of size 128:
Training epoch 1/10   lr=2.00000   error=0.130723
Training epoch 2/10   lr=2.00000   error=0.041368
Training epoch 3/10   lr=2.00000   error=0.036400
Training epoch 4/10   lr=1.00000   error=0.029529
Training epoch 5/10   lr=1.00000   error=0.027583
Training epoch 6/10   lr=1.00000   error=0.026163
Training epoch 7/10   lr=0.50000   error=0.024730
Training epoch 8/10   lr=0.50000   error=0.024051
Training epoch 9/10   lr=0.50000   error=0.023542
Training epoch 10/10   lr=0.25000   error=0.023005
Total process time: 4.066


In [11]:
outputMiniGSD = netMiniGD.predict(x_test ,y_test )


Test set: Avg. loss: 0.0253, Accuracy: 9149/10000 (91%)



P1Q2b:   
Please answer the following:
* Which implementation was faster?
* with lower training error?
* with better test accuracy?    

and explain how these results fit your answers in Q1A.

**Answer (P1Q2b).** Both networks have the same architecture, initialisation scheme, loss (MSE) and see the same 10,000 training images for 10 epochs; the only difference is the update rule. Results of the run above (Colab CPU; the numbers vary a little from run to run because of the random initialisation and shuffling):

| | per-sample GD (`fit`, lr = 0.05) | mini-batch SGD (`fit_mini_batch`, batch 128, step-decay lr 2.0 → 0.25) |
|---|---|---|
| parameter updates | 10,000 per epoch → 100,000 | 79 per epoch → 790 |
| training time (10 epochs) | 50.5 s | 4.1 s (≈ 12× faster) |
| final training error (MSE) | 0.0084 | 0.0230 |
| test accuracy (10,000 images) | 94.2 % (9419/10000) | 91.5 % (9149/10000) |

* **Faster:** mini-batch SGD, about 12× faster (4.1 s vs 50.5 s). Each update processes 128 samples as one matrix product `(128,784)·(784,128)` in optimised BLAS code instead of 128 separate `(1,784)·(784,128)` products driven from a Python loop, so the per-sample cost drops dramatically (this is the "vectorisation" advantage from P1Q1a); the number of Python-level iterations falls from 100,000 to 790.
* **Lower training error:** per-sample GD. In the same 10 epochs it performs 128× more parameter updates. With the mean-gradient normalisation of `mse_grad` one mini-batch step is a single, well-averaged step, whereas the per-sample variant takes 128 small noisy steps per batch; the noisy steps are individually worse but there are many more of them, so after a fixed number of *epochs* the per-sample version has descended further. This is the trade-off described in P1Q1a: for a fixed number of samples seen, large batches give fewer (better) updates, and the gain per extra sample in a batch diminishes ($1/\sqrt m$ noise reduction) - at $m=128$ we are already beyond the "critical batch size" for this small problem, so most of the batch is "wasted" on gradient accuracy we do not need. The learning-rate schedule (large initial steps to compensate for the few updates, then halving every 3 epochs so that the last epochs refine the solution) is what makes the 790 updates competitive at all - in a short sweep of the starting rate, lr0 = 0.5 reached only 87 % test accuracy, lr0 = 1 reached 89 %, lr0 = 2 reached 91 %, and lr0 = 4 diverged (the tanh units saturate and the error gets stuck at 1.13).
* **Better test accuracy:** per-sample GD (94.2 % vs 91.5 %), simply because it is further along in training - its *training* error is also lower, so this is an under-training effect of the mini-batch run, not a generalisation effect. The test loss of each network is close to its training loss, so neither over-fits.

How this fits P1Q1a: the experiment compares the two ends of the batch-size axis for a *fixed number of epochs*. The mini-batch version wins on computational efficiency (vectorised, few updates) and gives smoother, monotonically decreasing epoch errors, but it converges more slowly *per epoch* because it makes far fewer updates. Measured per unit of *compute*, however, mini-batch SGD is far ahead: in the 50 s that the per-sample run needs for 10 epochs the mini-batch network could run ≈ 120 epochs (about 9,500 updates) and would be expected to reach a lower error and a higher accuracy than the per-sample run (in PS1 the same kind of mini-batch training reached 98 % on the full training set). The right comparison is therefore accuracy vs. wall-clock time, in which the mini-batch method is the clear winner, as predicted in P1Q1a.

## PS2 Part 2: CNN



A Convolutional Neural Network (CNN) is a type of a deep neural network that can perform well at image recognition and analysis tasks.


> In this problem set we will add CNN compatibility to our numpy-based neural network from PS 1.

The main components needed to implement a CNN architecture:  

**Convolution layer:** In the convolutional layers, filters are applied to the image to identify patterns and features. Imagine a sliding window that moves across the image, detecting edges, shapes, and other building blocks.  

**Pooling layer:** Pooling layers downsample the data, reducing its complexity and computational cost. This helps prevent overfitting, where the model memorizes training data instead of learning generalizable patterns.  

**Reshape layer:**
Reshaping the output of the convolution and pooling layers into the fully connected layer.   


A USEFUL HINT:

*Keeping account of the input and output shapes along the CNN network is crucial for the model to work as intended*



Useful resource:  
* [CS231](https://cs231n.github.io/convolutional-networks/).
* [Convolutional Neural Network From Scratch](https://www.kaggle.com/code/lusfernandotorres/convolutional-neural-network-from-scratch).
* [CNN with PyTorch](https://www.kaggle.com/code/sdelecourt/cnn-with-pytorch-for-mnist)

#### Data transformations





(Here I added a reshape that I originally intended for you to apply at model time).

In [12]:
from keras.datasets import mnist
from keras.utils import to_categorical
# load MNIST from server
# Using a standard library (keras.datasets) to load the mnist data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [13]:
# training data : 60000 samples
# reshape and normalize input data
x_train = x_train.astype('float32')
x_train /= 255
x_train = x_train.reshape(x_train.shape[0], 1, x_train.shape[1],x_train.shape[2])
# One-hot encoding of the output.
# Currently a number in range [0,9]; Change into a vector of size 10
# e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
y_train = y_train.reshape(y_train.shape[0], 1, y_train.shape[1])

# same for test data : 10000 samples
x_test = x_test.astype('float32')
x_test /= 255
x_test = x_test.reshape(x_test.shape[0],1, x_test.shape[1],x_test.shape[2])
y_test = to_categorical(y_test)
y_test = y_test.reshape(y_test.shape[0],1, y_test.shape[1])

In [14]:
print("x_train shape",x_train.shape)
print("x_test shape",x_test.shape)
print("y_train shape",y_train.shape)
print("y_test shape",y_test.shape)

x_train shape (60000, 1, 28, 28)
x_test shape (10000, 1, 28, 28)
y_train shape (60000, 1, 10)
y_test shape (10000, 1, 10)


Unlike the multilayer perceptron (MLP) architecture, the input to the convolution net is not flattened. In our case, the size of a single `MNIST` datapoint is (1,28,28). The first dimension is the number of channels (e.g. RGB  for color images). During training the inputs and outputs often  change shape as they pass through the convolution, pooling, and the other layers of the network.

---

### A. Convolution Layer

Here you are asked to implement a basic Convolution layer with the following components:



1.   Layer initialization - initialize the layer's filters using Normalized Xavier/Hu weights
2.   A generator function that yields all the available sliding windows in the input image.
3.   Forward propogation for the convolution layer
4.   Backward propogation for the convolution layer

[CS231](https://cs231n.github.io/convolutional-networks/#conv) provides a great overview both mathematically and visually,   
and [CS230](https://cs230.stanford.edu/section/4/) provides a good overview about Xavier initialization.




P2Q1a:  (10pts).   
Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.    
You can deviate slightly from the recommended structure below, but please remain true to the class structure and compatibility with the rest of the network classes and functions. The idea here is to write YOUR OWN version of CNN, so please try not to google the answers.





In [15]:
### SOLUTION IN THIS CODE BLOCK

class Convolution:
    # convolution layer using num_filters x num_filters filters.
    # size is the length/width of the filter window with shape (size x size)
    # In this simple implementation, there is no padding.

    def __init__(self, num_filters=3, size=3, activation=None):
        self.stride = 1 # for simplicity, we keep the stride=1.
        self.size = size
        self.activation = activation
        self.last_input = None

        # Initialize the Convolution layer filters
        ## SOLUTION:
        # Normalized Xavier (Glorot) uniform initialization:
        #   W ~ U(-l, l),  l = sqrt(6 / (fan_in + fan_out))
        # For a conv filter: fan_in  = (#input channels) * size * size  (1 channel for MNIST)
        #                    fan_out = num_filters * size * size
        # (He init, randn * sqrt(2/fan_in), is the ReLU-specific alternative.)
        fan_in = size * size
        fan_out = num_filters * size * size
        limit = np.sqrt(6.0 / (fan_in + fan_out))
        self.filters = np.random.uniform(-limit, limit, size=(num_filters, size, size))

    def patch_generator(self, image):
        # This generator (using yield) returns all the  possible patches of  the input image
        # of shape  (h,w)  for each channel c, together with the relevant y,x coordinates of the patch.
        # input_data/image must be shaped as (c,y,x), where c is the channel/filter
        ch,h,w = image.shape

        for y in range(0,h-self.size+1):
            for x in range(0,w-self.size+1,self.stride):
                ## SOLUTION:
                patch = image[:, y:y + self.size, x:x + self.size]   # shape (c, size, size)
                yield patch, y, x

    def forward_propagation(self, image):
        # Calculate the convolution forward pass using the filters together with a
        # user specified activation function.

        # If this is the first convolution layer, the input image is size (h,w) and must be expanded to (c,h,w).
        if len(image.shape)==2:
            image = np.expand_dims(image, axis=0)
        # keep track of last input for later backward propagation
        self.last_input = image
        # Initializing the output array:
        ## SOLUTION:
        input_dimension = image.shape[1]     # the dimension (height = width) of the input image
        # no padding, stride 1:  out = (in - size) / stride + 1
        output_dimension = (input_dimension - self.size) // self.stride + 1

        # compute output dimensions after the convolution layer
        output = np.zeros((self.filters.shape[0], output_dimension, output_dimension))

        # forward pass on each filter.
        # All filters are applied to a patch at once with one tensordot; this is equivalent to
        # the per-filter loop  output[f,y,x] = np.sum(patch * self.filters[f])  but saves a
        # factor of num_filters in Python iterations.
        for patch, y, x in self.patch_generator(image):
            ## SOLUTION:
            # output[f, y, x] = sum_c sum_ij patch[c,i,j] * filters[f,i,j]
            output[:, y, x] = np.tensordot(self.filters, patch.sum(axis=0), axes=([1, 2], [0, 1]))

        # Applying activation function (this is a small cheat/simplification
        #  because the activation should be a separate layer in itself.  )
        if self.activation is not None:
            output = self.activation.forward_propagation(output)
        return output

    def backward_propagation(self, grad_out, learning_rate):
        # Calculate the SGD in-gradients of the filters and the layer.
        # Update only the filters gradients.

        # back propagate through activation (again, a bit of a cheat)
        if self.activation is not None:
            grad_out = self.activation.backward_propagation(grad_out,learning_rate)

        ## SOLUTION:
        grad_in = np.zeros(self.last_input.shape)      # loss gradient of the input   (c, h, w)
        grad_filter = np.zeros(self.filters.shape)      # loss gradient of the filters (f, size, size)

        # out[f,y,x] = sum_ij patch_yx[i,j] * W[f,i,j]   =>
        #   dL/dW[f]            += dL/dout[f,y,x] * patch_yx      (summed over all positions y,x)
        #   dL/dpatch_yx[c,i,j] += dL/dout[f,y,x] * W[f,i,j]      (summed over all filters f)
        for patch, y, x in self.patch_generator(self.last_input):
            ## SOLUTION:
            g = grad_out[:, y, x]                                   # (f,) output gradients at this position
            grad_filter += g[:, None, None] * patch.sum(axis=0)     # gradient of every filter
            grad_in[:, y:y + self.size, x:x + self.size] += np.tensordot(g, self.filters, axes=(0, 0))  # gradient of input

        ## SOLUTION:
        self.filters -= learning_rate * grad_filter   # update filters using SGD

        return grad_in    # return the loss gradient for this layer's inputs

### B. Max-pooling Layer

As with the convolution layer, here you are asked to implement a basic max-pooling layer with the following components:


1.   A generator function that yields all the available sliding windows in the input image. You can reuse the generator from (1).
2.   Forward propogation for the max-pooling layer
3.   Backward propogation for the max-pooling layer

The pooling layer is responsible for reducing the dimensionality of the input. As with the convolution layer, the pooling layer slides a filter across the entire image input to calculate the output. Note that this layer does not have any weights.  As the filter slides through the input image, it selects the pixel with the maximum value to populate the output array.

P2Q1b: (10pts).       

Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.  





In [16]:
### SOLUTION IN THIS BLOCK

class MaxPool:
    def __init__(self,  size=2):
        # max pooling layer

        self.last_input = None
        self.stride = 1
        self.size = size

    def patch_generator(self, image):
        # returns a generator (using yield) of all  possible patches of size size x size
        # input_data/image must be shaped as (c,y,x)
        ch,h,w = image.shape

        for c in range(ch):
          for y in range(0,h-self.size+1):
              for x in range(0,w-self.size+1,self.stride):
                  patch = image[c, y:y + self.size, x:x + self.size]   # shape (size, size)
                  yield patch, c, y, x

    def forward_propagation(self, image):
        # keep track of last input for later backward propagation
        self.last_input = image

        # compute output dimensions after the max pooling
        num_channels, h_prev, w_prev = image.shape
        ## SOLUTION:
        h_new = (h_prev - self.size) // self.stride + 1
        w_new = (w_prev - self.size) // self.stride + 1
        # hold the values of the max pooling
        output = np.zeros((num_channels, h_new, w_new))

        # Calculate the max pool on all the image patches using the generator:
        for patch, c, y, x in self.patch_generator(image):
            output[c ,y, x] = patch.max()      # (ndarray method: much less call overhead than np.max in a 7k-iteration loop)
        return output

    def backward_propagation(self, grad_out, learning_rate):
        # Calculate the gradients of the last input image
        num_channels, orig_dim, *_ = self.last_input.shape
        grad_in = np.zeros(self.last_input.shape)

        # Using the generator, go over all the channels and calcualte the max-pooling input gradients.
        # Only the element that attained the max in each window receives the gradient
        # (d max / d x_i = 1 for the arg-max element, 0 otherwise); overlapping windows accumulate.
        for patch, c, y, x in self.patch_generator(self.last_input):
            # (the template used np.nanargmax; the data has no NaNs and the plain argmax is ~20x cheaper per call)
            (i, j) = np.unravel_index(patch.argmax(), patch.shape)   # offset of the max inside the patch
            grad_in[c, y + i, x + j] += grad_out[c, y, x]

        return grad_in

### C. Reshape Layer

A reshape layer in a neural network is a transformation layer that doesn't perform any computations itself (no weights).   
Its purpose is to simply change the dimensionality (shape) of the data flowing through the network.

Here's how reshape layers work:

You define a target shape for the output of the reshape layer.   
This target shape specifies the number of elements along each dimension (width, height, channels, etc.) for the new data format.

P2Q1c:  (5pts).   
Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.  





In [17]:
### SOLUTION IN THIS BLOCK ####
class Reshape:
    def __init__(self,input_shape,output_shape):
        self.input_shape=input_shape
        self.output_shape=output_shape
    def forward_propagation(self,input_data):
        return np.reshape(input_data, self.output_shape)

    def backward_propagation(self,output_gradient,learning_rate):
        # you can ignore the learning_rate
        return np.reshape(output_gradient, self.input_shape)

### Testing Your Neural Network

### Defining our main neural network architecture

Define your CNN network's architecture:

* A convolution layer with some filters `(f)` and a window size of `(w x w)`, with a `tanh` activation function.
* A max pooling layer of a window size of `(p x p)`.  
* First affine layer that takes your input and outputs 128 nodes
* `tanh` activation layer following the first affine layer
* Second affine layer that takes the first layer's input and outputs 64 nodes
* `tanh` activation layer following the second affine layer
* Third affine layer that takes your second layer's input and outputs nodes in the size of the Y labels.
* `tanh` activation layer following the last affine layer


You can compute the shapes of the convolution and max-pool layers' outputs  as a function of the input shape, the convolution sliding window size, and the number of filters for applied. ( the stride and padding are usually also used to calculate the output shapes, but in this simple implementation we set the stride to 1 with zero padding.).

P2Q2a: (5pts).        
Calculate the input and output shapes of each layer in the network below, up until the first affine layer (including).  (mathematically) explain your answer.  

**Answer (P2Q2a).**

With no padding ($P=0$) and stride $S=1$, a window of size $K$ sliding over an input of spatial size $H\times W$ fits in
$H_{out}=\left\lfloor\frac{H+2P-K}{S}\right\rfloor+1 = H-K+1$ positions (and likewise for $W$): the window's top-left corner can be at rows $0,\dots,H-K$.
The number of output channels of a convolution layer equals its number of filters $F$ (each filter produces one feature map); pooling and reshaping keep the number of channels / the number of elements.

| layer | input shape | output shape | formula |
|---|---|---|---|
| input image | - | $(1,28,28)$ | 1 channel (grey-scale), $28\times28$ pixels |
| `Convolution(num_filters=12, size=3)` + ReLU | $(1,28,28)$ | $(12,26,26)$ | $H_{out}=28-3+1=26$, channels $=F=12$ (ReLU is element-wise) |
| `MaxPool(size=3)` (stride 1) | $(12,26,26)$ | $(12,24,24)$ | $H_{out}=26-3+1=24$, channels unchanged |
| `Reshape((12,24,24),(1,6912))` | $(12,24,24)$ | $(1,6912)$ | $12\cdot24\cdot24=6912$ elements, same count |
| `Affine_Layer(6912,128)` | $(1,6912)$ | $(1,128)$ | $y=xW+b$, $W\in\mathbb R^{6912\times128}$ |

So $c=12$, $y_{new}=x_{new}=24$. Number of trainable parameters: the convolution has only $12\cdot3\cdot3=108$ weights, whereas the first affine layer has $6912\cdot128+128=884{,}864$ - because the stride-1 pooling barely reduces the resolution ($26\to24$), the fully-connected layer dominates the parameter count. (The remaining layers: $(1,128)\to(1,64)\to(1,10)$.) The computation is verified in the code cell below.

In [18]:
# P2Q2a - computing the shapes of each layer as a function of the hyper-parameters
def conv_out_size(n_in, kernel, stride=1, padding=0):
    # generic formula: floor((n_in + 2*padding - kernel) / stride) + 1
    return (n_in + 2 * padding - kernel) // stride + 1

in_channels, H = 1, 28           # MNIST: (1, 28, 28)
f, w = 12, 3                     # number of filters, convolution window (w x w)
p = 3                            # max-pool window (p x p), stride 1

H_conv = conv_out_size(H, w)     # 28 - 3 + 1 = 26
H_pool = conv_out_size(H_conv, p)  # 26 - 3 + 1 = 24
n_flat = f * H_pool * H_pool     # 12 * 24 * 24 = 6912

print(f"input            : {(in_channels, H, H)}")
print(f"after Convolution: {(f, H_conv, H_conv)}   (H_out = H - w + 1)")
print(f"after MaxPool    : {(f, H_pool, H_pool)}   (H_out = H - p + 1)")
print(f"after Reshape    : {(1, n_flat)}")
print(f"after Affine(1)  : {(1, 128)}")
print(f"parameters: conv = {f*w*w}, first affine = {n_flat*128 + 128:,}")

# empirical check with a single image
_conv, _pool = Convolution(num_filters=f, size=w), MaxPool(size=p)
_a = _conv.forward_propagation(x_train[0]); _b = _pool.forward_propagation(_a)
print("empirical:", x_train[0].shape, "->", _a.shape, "->", _b.shape, "->",
      Reshape(_b.shape, (1, n_flat)).forward_propagation(_b).shape)

input            : (1, 28, 28)
after Convolution: (12, 26, 26)   (H_out = H - w + 1)
after MaxPool    : (12, 24, 24)   (H_out = H - p + 1)
after Reshape    : (1, 6912)
after Affine(1)  : (1, 128)
parameters: conv = 108, first affine = 884,864
empirical: (1, 28, 28) -> (12, 26, 26) -> (12, 24, 24) -> (1, 6912)


P2Q2b:  (5pts).   

Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the network architecture works properly.  





In [19]:
#### SOLUTION IN THIS BLOCK (in  reshape parameters) ####

# Network Architecture
CNNnet = MyNetwork()
num_classes = 10
# Reshape sizes:
## SOLUTION
c = 12                       # output channels = number of filters
y_new = (28 - 3 + 1) - 3 + 1 # 24: conv  28 -> 26 (size 3, stride 1, no padding), max-pool 26 -> 24 (size 3, stride 1)
x_new = (28 - 3 + 1) - 3 + 1 # 24 (square images)

# Adding convolution and max-pool layers:
CNNnet.add(Convolution(num_filters=12, size=3,
                    activation=ActivationLayer(relu, relu_grad))) #
CNNnet.add(MaxPool(size=3))

# Reshaping for first affine layer:
CNNnet.add(Reshape((c,y_new,x_new),(1,c*y_new*x_new)))

# Adding affine layers and activation functions:
CNNnet.add(Affine_Layer(c*y_new*x_new, 128))
CNNnet.add(ActivationLayer(tanh, tanh_grad))
CNNnet.add(Affine_Layer(128, 64))
CNNnet.add(ActivationLayer(tanh, tanh_grad))
CNNnet.add(Affine_Layer(64, num_classes))
CNNnet.add(ActivationLayer(tanh, tanh_grad))

# CNNnet.add(ActivationLayer(relu, relu_grad))
# CNNnet.add(ActivationLayer(tanh, tanh_grad))
# CNNnet.add(ActivationLayer(sigmoid, sigmoid_grad))

### Training!

There is no need to train the model on the entire dataset (you'l see why now...). Try to prove that your network works as expected by showing that your error decreases dramatically even on a small sample (say, around 1k).

In [20]:
# While developing, it is recommended to train your model on a subset of the data... / or low epochs.
# Training will be pretty slow if we update at each iteration on 60000 samples...]
CNNnet.use_loss(mse, mse_grad)
epoch_num = 5   # (the template suggested 20; 5 epochs on 1000 samples already clear 85% test accuracy, see below)
lr = 0.1
t1 = time.time()
CNNnet.fit(x_train[:1000], y_train[:1000], epochs=epoch_num,
           learning_rate=lr)
print(f"Total process time: {round(time.time() - t1,3)}")

Training on 1,000 samples:
Training epoch 1/5   error=0.071517
Training epoch 2/5   error=0.039610
Training epoch 3/5   error=0.030078
Training epoch 4/5   error=0.024067
Training epoch 5/5   error=0.019697
Total process time: 735.546


Hurray - if you got to this point, your CNN model works, but is EXTREMELY slow.   
Let's check why using a cProfiler:

In [21]:
# Profiling a tiny run (20 samples, 1 epoch) - enough to see where the time goes:
CNNnet.prof(x_train[:20], y_train[:20], epochs=1, learning_rate=lr)

Training on 20 samples:
Training epoch 1/1   error=0.014948
         1876325 function calls (1876294 primitive calls) in 4.830 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    27080    0.033    0.000    0.033    0.000 3573461534.py:26(patch_generator)
       20    0.065    0.003    0.562    0.028 3573461534.py:38(forward_propagation)
       20    0.245    0.012    0.829    0.041 3573461534.py:71(backward_propagation)
       20    0.000    0.000    0.001    0.000 3771341627.py:6(forward_propagation)
       20    0.000    0.000    0.001    0.000 3771341627.py:9(backward_propagation)
   276520    0.335    0.000    0.335    0.000 469114053.py:11(patch_generator)
       20    0.160    0.008    0.903    0.045 469114053.py:22(forward_propagation)
       20    1.393    0.070    2.178    0.109 469114053.py:39(backward_propagation)
        4    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
        8    0.000

P2Q2c:  (5pts).     
Why do you think our CNN implementation is slow? Your answer should relate to the  approximate `O(n)` time/computational complexity .

**Answer (P2Q2c).**

The *arithmetic* is small. Per image, the convolution needs $F\cdot H_{out}^2\cdot K^2 = 12\cdot26^2\cdot3^2\approx 73{,}000$ multiply-adds (forward) and about twice that in the backward pass, the pooling needs $C\cdot H_{out}^2\cdot P^2 = 12\cdot24^2\cdot 9\approx 62{,}000$ comparisons, and the first affine layer $6912\cdot128\approx 885{,}000$ multiply-adds. Altogether a few million flops per sample, which a CPU does in about a millisecond when it is expressed as large matrix operations. Our implementation nevertheless needed 735.5 s for 5 epochs on 1,000 images in the run above, i.e. ≈ 147 ms per sample and ≈ 2.5 minutes per epoch. At that rate the suggested 20 epochs on 1k images would take ≈ 49 min, and one epoch over the full 60k training set ≈ 2.5 h. The reason is **where the loops are executed**, i.e. the constant factor hidden in the $O(\cdot)$:

* The sliding windows are enumerated in **Python**. For every image the convolution generator yields $H_{out}^2=676$ patches (forward *and* backward) and the pooling generator $C\cdot H_{out}^2=12\cdot576=6{,}912$ patches (forward and backward). So the time per image is $O\big(H_{out}^2 + C\,H_{pool}^2\big)\approx 15{,}000$ **interpreted iterations**, each doing a slice, a `yield`, a tuple unpack and 1-3 numpy calls on a tiny $3\times3$ array. A numpy call costs ≈ 1-10 µs of overhead regardless of the array size, so 15k iterations × several calls ≈ 100-150 ms, matching the measured 147 ms per sample; the actual arithmetic is negligible. The profile above confirms it: essentially all of the time is in `MaxPool.backward_propagation` / `MaxPool.forward_propagation` and `Convolution.backward_propagation` / `forward_propagation`, spread over ~10⁵ calls of `argmax`, `max`, `tensordot`, `unravel_index` and the generators, whereas the three affine layers (which hold 99 % of the parameters and most of the flops) take < 7 ms per sample in a per-layer timing because each is a single `np.dot`. (The template's `np.nanargmax` is even worse - ≈ 20 µs per call because it first copies the patch to look for NaNs - replacing it by `patch.argmax()` alone made the layer ≈ 4× faster.)
* The training loop is **per sample**: `fit` performs one forward + backward pass and one parameter update for every image, so the Python overhead above is paid $N\cdot E$ times with no batching: total cost $O\big(N\,E\,(H_{out}^2 + C H_{pool}^2)\big)$ Python iterations $\approx 1000\cdot20\cdot 15{,}000 = 3\cdot10^8$ for the suggested run. A batched implementation would replace the $N$ factor by $N/m$ large array operations.
* The architecture itself is also wasteful: stride-1 pooling with a $3\times3$ window reduces the resolution only from 26 to 24 ($\times 0.85$ in each dimension) instead of the usual $\times 0.5$ of a stride-2 pool, so the pooling layer has to visit $12\cdot24^2$ windows, the affine layer has 6,912 inputs (885k weights), and no spatial information is actually discarded.

In asymptotic terms the per-image complexity, $O(F H_{out}^2 K^2 + C H_{pool}^2 P^2 + n_{flat}\cdot128)$, is exactly the same as PyTorch's; what differs is the constant: PyTorch (and a vectorised numpy version using `im2col`/`sliding_window_view` + one `tensordot`, or `as_strided` for the pooling) executes the $O(H_{out}^2)$ inner loops in compiled C/BLAS (and in parallel on many cores or a GPU) with one Python call per layer per *batch*, i.e. ≈ 10⁴ times fewer interpreter round-trips. That is why the identical PyTorch architecture below trains on 60,000 images per epoch faster than ours does on 1,000.

### Evaluation

Exciting! Now is the time to test your model.     


    May the gradients be always in your favor.

P2Q2b (cont'd):

**Try to limit the number of epochs and the size of the trained data, while still achieving an accuracy score of >85%.**

In [22]:
output = CNNnet.predict(x_test[:500] ,y_test[:500] )


Test set: Avg. loss: 0.0356, Accuracy: 444/500 (89%)



### Benchmarking against PyTorch

How well your model performs against a similar-architecture PyTorch model?   
It is time to find out:

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset

**Prepare the data as tensors using PyTorch DataLoader:**

In [24]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# # training data : 60000 samples
# # reshape and normalize input data
# x_train = x_train.reshape(x_train.shape[0], 1, 28*28) not needed
x_train = x_train.astype('float32')
x_train /= 255
# # One-hot encoding of the output.
# # Currently a number in range [0,9]; Change into a vector of size 10
# # e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
# # same for test data : 10000 samples
# x_test = x_test.reshape(x_test.shape[0], 1, 28*28) not needed
x_test = x_test.astype('float32')
x_test /= 255
y_test = to_categorical(y_test)

In [25]:
t_train =  TensorDataset(torch.Tensor(x_train),torch.Tensor(y_train))
t_test =  TensorDataset(torch.Tensor(x_test),torch.Tensor(y_test))
train_loader = torch.utils.data.DataLoader(dataset=t_train, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=t_test, batch_size=64, shuffle=False)

Define a `PyTorchCNN` class with an identical architecture you used in your home-made network.  
Hint: use `x = x.view()` to handle reshapes.

P2Q3a: (5pts).   

In [26]:
#### SOLUTION IN THIS BLOCK ####

class PyTorchCNN(nn.Module):
    def __init__(self):
        super(PyTorchCNN, self).__init__()
        input_size = 28                        # MNIST images are 28 x 28, 1 channel
        num_classes = 10
        num_filters, conv_size, pool_size = 12, 3, 3     # identical to the numpy CNNnet
        # Same shape arithmetic as in the home-made network (stride 1, no padding):
        conv_out = input_size - conv_size + 1  # 26
        pool_out = conv_out - pool_size + 1    # 24
        self.flat_size = num_filters * pool_out * pool_out   # 12 * 24 * 24 = 6912

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=num_filters, kernel_size=conv_size, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=pool_size, stride=1)   # stride 1 like our MaxPool
        self.fc1 = nn.Linear(self.flat_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.view(-1, 1, 28, 28)        # (batch, 28, 28) -> (batch, channels=1, 28, 28)
        x = F.relu(self.conv1(x))        # -> (batch, 12, 26, 26)   Convolution + ReLU
        x = self.pool(x)                 # -> (batch, 12, 24, 24)   MaxPool(3), stride 1
        x = x.view(-1, self.flat_size)   # -> (batch, 6912)         Reshape
        x = torch.tanh(self.fc1(x))      # -> (batch, 128)
        x = torch.tanh(self.fc2(x))      # -> (batch, 64)
        x = torch.tanh(self.fc3(x))      # -> (batch, 10)           tanh output, as in CNNnet (MSE loss)
        return x

In [27]:

# Train the model
num_epochs = 10
pt_learning_rate = 0.01
pt_network = PyTorchCNN()
optimizer = torch.optim.Adam(pt_network.parameters(), lr=pt_learning_rate)
criterion = nn.MSELoss()

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Forward pass
        # labels = labels.view(64,1,10)
        outputs = pt_network(images)
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # A handy printout:
        if (i + 1) % 500 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')


Epoch [1/10], Step [500/938], Loss: 0.0100
Epoch [2/10], Step [500/938], Loss: 0.0096
Epoch [3/10], Step [500/938], Loss: 0.0119
Epoch [4/10], Step [500/938], Loss: 0.0061
Epoch [5/10], Step [500/938], Loss: 0.0141
Epoch [6/10], Step [500/938], Loss: 0.0089
Epoch [7/10], Step [500/938], Loss: 0.0072
Epoch [8/10], Step [500/938], Loss: 0.0090
Epoch [9/10], Step [500/938], Loss: 0.0069
Epoch [10/10], Step [500/938], Loss: 0.0109


Evaluation:

In [28]:
pt_network.eval()
test_losses = []
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        output = pt_network(data)
        test_loss += criterion(output, target,)
        pred = output.data.max(1, keepdim=True)[1]
        correct += pred.eq(target.data.max(1,keepdim=True)[1]).sum()

test_loss /= len(test_loader.dataset)
test_losses.append(test_loss)
print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
  test_loss, correct, len(test_loader.dataset),
  100. * correct / len(test_loader.dataset)))


Test set: Avg. loss: 0.0001, Accuracy: 9729/10000 (97%)



P2Q3b (5pts).   

Time for some questions:
1. Which one of the models performed better? Why?
2. Which one of the models performed faster? Why?  
3. What would you change in your network's architecture?   
4. What would you change in your model's solution algorithm?

**Answers (P2Q3b).**

1. **Which model performed better? Why?**
   The PyTorch model: 97.3 % (9729/10000) on the full 10,000-image test set after 10 epochs on all 60,000 training images, versus 88.8 % (444/500) on the first 500 test images for the numpy CNN trained on only 1,000 images for 5 epochs. The architectures are identical (12 filters 3×3 + ReLU → 3×3 stride-1 max-pool → 6912-128-64-10 with tanh, MSE loss), so the difference is not in the model but in the *amount of training and the optimiser*: the PyTorch network saw 60× more data (600k sample presentations vs 5k), was trained with mini-batches of 64 and the Adam optimiser (per-parameter adaptive step sizes with momentum) instead of plain per-sample SGD with a fixed learning rate, and uses biases in the convolution and a proper (Kaiming-uniform) initialisation of every layer. Given the same data budget the two models would be expected to behave similarly - our network already reaches 88.8 % from 1k images, which shows that the hand-written convolution/pooling gradients are correct.

2. **Which model was faster? Why?**
   PyTorch, by orders of magnitude. Our implementation needed 735.5 s for 5 epochs on 1,000 images, i.e. 5,000 image presentations at ≈ 147 ms each. The PyTorch run made 600,000 image presentations (10 epochs × 60,000 images); at our speed that would take ≈ 24.5 hours, while the PyTorch training cell finished within an ordinary Colab cell run (the cell does not print its own time). The reasons were analysed in P2Q2c: PyTorch runs the sliding-window loops in compiled, multi-threaded C++ kernels (`conv2d` is lowered to a single GEMM via im2col / Winograd, `max_pool2d` is one kernel), processes a mini-batch of 64 images per call, so the Python interpreter is entered a handful of times per batch instead of ~15,000 times per image, and it uses optimised BLAS (or a GPU if one is available). Our code pays a few micro-seconds of interpreter/numpy overhead per 3×3 patch, and updates the parameters after every single image.

3. **What would you change in the network's architecture?**
   * Use **stride-2 (or 2×2 non-overlapping) pooling** - the current 3×3 stride-1 pool leaves 24×24×12 = 6,912 features, so the first affine layer holds 885k of the ≈ 894k parameters and dominates both compute and over-fitting risk; a 2×2/stride-2 pool would give 13×13×12 = 2,028 features (3.4× fewer weights) while discarding no useful information.
   * Add a **second conv + pool block** (e.g. 12→24 filters) so that the network learns hierarchical features and the fully-connected part becomes small; use **padding** to keep the border pixels; add **biases** to the convolution.
   * Replace the **tanh output + MSE** by a **softmax output with cross-entropy loss**: the gradient $\hat y - y$ does not vanish when the output saturates, which is the main thing holding back the MSE/tanh combination (as already observed in PS1).
   * Use **ReLU + He initialisation** in the hidden affine layers, optionally **batch-normalisation** after the convolution and **dropout** before the last layer for regularisation.

4. **What would you change in the model's solution algorithm?**
   * **Vectorise**: implement the convolution with `im2col` / `np.lib.stride_tricks.sliding_window_view` so that one `tensordot`/matrix product computes all windows and all filters at once (and the same `as_strided` trick + `max` over the last two axes for pooling, with a boolean mask for the backward pass). This removes the ~15,000 Python iterations per image and gives a ~100× speed-up in pure numpy.
   * **Mini-batch training** with the `fit_mini_batch` from Part 1 (process 64-128 images per update, shuffle each epoch) - the layers would need a leading batch dimension, after which the whole forward/backward pass becomes a few large array operations and the gradient noise is reduced as discussed in P1Q1a.
   * A better **optimiser and schedule**: momentum / Adam instead of plain SGD, and a learning-rate decay (step or cosine) instead of the fixed lr = 0.1, plus early stopping on a validation set.
   * Train on **all 60,000 images** (feasible once the code is vectorised), with simple **data augmentation** (small shifts/rotations) to improve generalisation.
   Together these changes would make the numpy CNN both faster and at least as accurate as the PyTorch benchmark on MNIST (≈ 98-99 %).